In [55]:
import pandas as pd

In [62]:
qs = pd.read_json('mintaka_train.json')

In [63]:
qs.head()

,id,question,translations,questionEntity,answer,category,complexityType
0,a9011ddf,What is the seventh tallest mountain in North ...,"{'ar': 'ما سابع أعلى جبل في أمريكا الشمالية؟',...","[{'name': 'Q49', 'entityType': 'entity', 'labe...","{'answerType': 'entity', 'answer': [{'name': '...",geography,ordinal
1,2723bb1b,Which actor was the star of Titanic and was bo...,{'ar': 'مَنْ الممثل الذي لعب دور البطولة في في...,"[{'name': 'Q44578', 'entityType': 'entity', 'l...","{'answerType': 'entity', 'answer': [{'name': '...",movies,intersection
2,88349c89,Which actor starred in Vanilla Sky and was mar...,{'ar': 'مَنْ الممثل الذي لعب دور البطولة في في...,"[{'name': 'Q174346', 'entityType': 'entity', '...","{'answerType': 'entity', 'answer': [{'name': '...",movies,intersection
3,bff78c91,What year was the first book of the A Song of ...,"{'ar': 'في أي عام تم نشر أول كتاب من سلسلة ""أغ...","[{'name': 'Q45875', 'entityType': 'entity', 'l...","{'answerType': 'date', 'answer': ['1996'], 'me...",books,generic
4,982450cf,Who is the youngest current US governor?,"{'ar': 'مَن أصغر حاكم ولاية أمريكي حالٍ؟', 'de...","[{'name': 'Q889821', 'entityType': 'entity', '...","{'answerType': 'entity', 'answer': [{'name': '...",politics,superlative


In [64]:
qs = qs[:50]

In [65]:
qs.drop(columns=['id', 'category', 'complexityType', 'translations'], inplace=True)

In [67]:
qs.drop([27, 48], inplace=True)




In [68]:
qs.head()


,question,questionEntity,answer
0,What is the seventh tallest mountain in North ...,"[{'name': 'Q49', 'entityType': 'entity', 'labe...","{'answerType': 'entity', 'answer': [{'name': '..."
1,Which actor was the star of Titanic and was bo...,"[{'name': 'Q44578', 'entityType': 'entity', 'l...","{'answerType': 'entity', 'answer': [{'name': '..."
2,Which actor starred in Vanilla Sky and was mar...,"[{'name': 'Q174346', 'entityType': 'entity', '...","{'answerType': 'entity', 'answer': [{'name': '..."
3,What year was the first book of the A Song of ...,"[{'name': 'Q45875', 'entityType': 'entity', 'l...","{'answerType': 'date', 'answer': ['1996'], 'me..."
4,Who is the youngest current US governor?,"[{'name': 'Q889821', 'entityType': 'entity', '...","{'answerType': 'entity', 'answer': [{'name': '..."


In [69]:
res = pd.read_csv('LLMResults.csv')

In [70]:
res.head()

,Unnamed: 0,SAE Question,AAVE Question,SPARQL,Results,Attempts,Name,Answer
0,0,What is the seventh tallest mountain in North ...,Wha's da seventh tallest mountain in North Ame...,SELECT DISTINCT ?item ?itemLabel ?elev WHERE {...,"{'head': {'vars': ['item', 'itemLabel', 'elev'...",1,['El Diente Peak'],El Diente Peak
1,1,Which actor was a star of Titanic and was born...,Which actor was da star of Titanic and was bor...,SELECT ?item ?itemLabel WHERE {\n wd:Q1258 wd...,"{'head': {'vars': ['item', 'itemLabel']}, 'res...",1,NaN,I don't know
2,2,Which actor starred in Vanilla Sky and was mar...,Which actor starred in Vanilla Sky an' was mar...,SELECT ?actor ?actorLabel WHERE {\n ?film wdt...,"{'head': {'vars': ['actor', 'actorLabel']}, 'r...",1,NaN,Nan Chan
3,3,In what year did the first book of the A Song ...,What year da first book of A Song of Ice and F...,SELECT (MIN(YEAR(?pub)) AS ?year) WHERE {\n ?...,"{'head': {'vars': ['year']}, 'results': {'bind...",1,[],2000
4,4,Who is the youngest current U.S. governor?,Who da youngest current US governor?,SELECT ?person ?personLabel ?stateLabel ?dob W...,"{'head': {'vars': ['person', 'personLabel', 's...",1,"['Sarah Sanders', 'Arkansas']",Sarah Sanders


In [90]:
pred = res['Answer'].tolist()
norm = qs['answer'].tolist()

In [123]:
import json
gold = []
def normalize(norm):
    for row in norm:
        if isinstance(row, str):
            row = json.loads(row)

        try:
            if row["answerType"] in {"boolean", "date", "numerical"}:
                gold.append(row["answer"])
            else:
                gold.append(row["answer"][0]["label"]["en"])
        except (KeyError, IndexError, TypeError):
            gold.append(None)

    return gold


In [124]:
gold = normalize(norm)


In [132]:
import ast

def strip_brackets(x):
    if isinstance(x, str) and x.startswith("[") and x.endswith("]"):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list) and len(parsed) == 1:
                return parsed[0]
        except (ValueError, SyntaxError):
            pass
    return x

def unwrap(x):
    if isinstance(x, list) and len(x) == 1:
        return x[0]
    return x



In [140]:
gold = strip_brackets(gold)

In [144]:
gold = [str(unwrap(item)) for item in gold]



In [156]:
def replace_true(lst):
    for i, x in enumerate(lst):
        if x == 'True':
            lst[i] = 'yes'
    return lst

def replace_false(lst):
    for i, x in enumerate(lst):
        if x == 'False':
            lst[i] = 'no'
    return lst



In [157]:
gold = replace_true(gold)
gold = replace_false(gold)

In [165]:
import collections
from typing import List, Any

def calculate_f1(pred: List[Any], gold: List[Any]) -> float:
    if not pred or not gold:
        return float(pred == gold)

    pred = [str(x) for x in pred]
    gold = [str(x) for x in gold]

    common = collections.Counter(pred) & collections.Counter(gold)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred)
    recall = num_same / len(gold)
    return (2 * precision * recall) / (precision + recall)


In [182]:
def find_unanswered(pred):
    counter =0
    for item in pred:
        if item =="I don't know":
            counter +=1
    return (counter /len(pred)) * 100
        

In [ ]:
def calculate_em(pred: Union[list, str, None], answer: Union[list, str, None], mode: str) -> int:
    """
    Calculate an exact match score
    Args:
        pred: predicted answer from a model
        answer: answer from the Mintaka test set
        mode: mode of evaluation (kg or text)
    Returns:
        1 if the prediction exactly matches the answer, else 0
    """
    if mode == 'text' and pred and answer:
        pred = normalize_and_tokenize_text(pred)
        answer = normalize_and_tokenize_text(answer)
        for i in range(0, len(pred) - len(answer) + 1):
            if answer == pred[i: i + len(answer)]:
                return True
        return False
    else:
        return int(pred == answer)

In [198]:
mintaka_f1 = round(calculate_f1(pred,gold), 3) 
mintaka_unanswered = round(find_unanswered(pred), 3)

In [189]:
sparql = res['SPARQL'].tolist()

In [192]:
qs.iloc[0]['questionEntity']

[{'name': 'Q49',
  'entityType': 'entity',
  'label': 'North America',
  'mention': 'North America',
  'span': [40, 53]},
 {'name': 7, 'entityType': 'ordinal', 'mention': 'seventh', 'span': [12, 19]}]

In [193]:
sparql[0]

'SELECT DISTINCT ?item ?itemLabel ?elev WHERE {\n  ?item wdt:P31/wdt:P279* wd:Q8502;\n        wdt:P2044 ?elev.\n  { ?item wdt:P30 wd:Q49 } UNION { ?item wdt:P17/wdt:P30 wd:Q49 }.\n  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }\n}\nORDER BY DESC(?elev)\nOFFSET 6\nLIMIT 1'

In [195]:
res['Answer'][0]

'El Diente Peak'

In [197]:
res['SAE Question'][0]

'What is the seventh tallest mountain in North America?'

In [201]:
mintaka_f1


0.417

In [202]:
lm = pd.read_csv('LLMResults.csv')

In [203]:
print(lm.iloc[0]['SPARQL'])

SELECT DISTINCT ?item ?itemLabel ?elev WHERE {
  ?item wdt:P31/wdt:P279* wd:Q8502;
        wdt:P2044 ?elev.
  { ?item wdt:P30 wd:Q49 } UNION { ?item wdt:P17/wdt:P30 wd:Q49 }.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY DESC(?elev)
OFFSET 6
LIMIT 1
